# 🏥 BUSI Dataset — Week 1: Data Preparation
### Breast Ultrasound Image Segmentation + Classification Project

**এই notebook এ যা করব:**
- ✅ Dataset structure দেখব
- ✅ Duplicate image remove করব (SSIM দিয়ে)
- ✅ Train / Val / Test split করব (70/15/15)
- ✅ Augmentation pipeline তৈরি করব
- ✅ Dataset visualization করব

---

## Step 0: Library Install করুন

In [ ]:
!pip install -q scikit-image albumentations matplotlib seaborn tqdm

## Step 1: Dataset Upload করুন Google Drive থেকে

> **আপনার curated BUSI dataset Google Drive-এ রাখুন, তারপর নিচের cell run করুন।**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# আপনার dataset path এখানে দিন
# উদাহরণ: /content/drive/MyDrive/BUSI_Dataset/
DATASET_PATH = '/content/drive/MyDrive/BUSI_Dataset'  # ← আপনার path দিন

print(f'Dataset path set: {DATASET_PATH}')

## Step 2: Import এবং Dataset Structure দেখুন

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from itertools import combinations
from skimage.metrics import structural_similarity as ssim
import shutil
import json
import random

# Seed fix করুন — reproducibility এর জন্য
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('✅ Libraries imported successfully!')

In [ ]:
# Dataset structure দেখুন
def explore_dataset(dataset_path):
    """
    Dataset এর folder structure এবং image count দেখায়
    """
    dataset_path = Path(dataset_path)
    classes = ['benign', 'malignant', 'normal']
    
    print('📁 Dataset Structure:')
    print('=' * 50)
    
    total_images = 0
    total_masks  = 0
    class_info   = {}
    
    for cls in classes:
        cls_path = dataset_path / cls
        if not cls_path.exists():
            print(f'⚠️  Folder not found: {cls}')
            continue
        
        # Image এবং mask আলাদা করুন
        all_files = list(cls_path.glob('*.png')) + list(cls_path.glob('*.jpg'))
        images    = [f for f in all_files if 'mask' not in f.stem.lower()]
        masks     = [f for f in all_files if 'mask' in f.stem.lower()]
        
        class_info[cls] = {
            'images': images,
            'masks':  masks,
            'count':  len(images)
        }
        
        total_images += len(images)
        total_masks  += len(masks)
        
        print(f'  📂 {cls.upper()}')
        print(f'      Images : {len(images)}')
        print(f'      Masks  : {len(masks)}')
        print()
    
    print(f'  📊 Total Images: {total_images}')
    print(f'  📊 Total Masks : {total_masks}')
    
    return class_info

class_info = explore_dataset(DATASET_PATH)

In [ ]:
# Class distribution bar chart দেখুন
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

classes = list(class_info.keys())
counts  = [class_info[c]['count'] for c in classes]
colors  = ['#2ecc71', '#e74c3c', '#3498db']

# Bar chart
axes[0].bar(classes, counts, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Images')
for i, (c, v) in enumerate(zip(classes, counts)):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts, labels=classes, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Class Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Distribution chart saved!')

## Step 3: Sample Images দেখুন (Image + Mask)

In [ ]:
def show_samples(class_info, n_samples=2):
    """
    প্রতিটি class থেকে sample image + mask দেখায়
    """
    classes = list(class_info.keys())
    fig, axes = plt.subplots(len(classes), n_samples * 2,
                             figsize=(n_samples * 6, len(classes) * 3))
    
    for row, cls in enumerate(classes):
        images = class_info[cls]['images'][:n_samples]
        
        for col, img_path in enumerate(images):
            # Image load করুন
            img  = cv2.imread(str(img_path))
            img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Corresponding mask খুঁজুন
            mask_path = img_path.parent / (img_path.stem + '_mask' + img_path.suffix)
            
            # Image দেখান
            ax_img  = axes[row][col * 2]
            ax_mask = axes[row][col * 2 + 1]
            
            ax_img.imshow(img, cmap='gray')
            ax_img.set_title(f'{cls} — Image {col+1}', fontsize=9)
            ax_img.axis('off')
            
            if mask_path.exists():
                mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
                ax_mask.imshow(mask, cmap='gray')
                ax_mask.set_title(f'Mask {col+1}', fontsize=9)
            else:
                ax_mask.text(0.5, 0.5, 'No mask', ha='center', va='center')
                ax_mask.set_title('No Mask', fontsize=9)
            ax_mask.axis('off')
    
    plt.suptitle('Sample Images & Masks per Class', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()

show_samples(class_info, n_samples=2)

## Step 4: Duplicate Remove করুন (SSIM দিয়ে)

> SSIM score > 0.95 মানে দুটো image প্রায় একই → duplicate হিসেবে ধরব

In [ ]:
def find_duplicates(image_list, threshold=0.95, resize_to=(128, 128)):
    """
    SSIM দিয়ে duplicate image খুঁজে বের করে
    """
    print(f'🔍 Checking {len(image_list)} images for duplicates...')
    
    # সব image grayscale এ load করুন
    loaded = {}
    for img_path in tqdm(image_list, desc='Loading images'):
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, resize_to)
            loaded[str(img_path)] = img
    
    # সব pair compare করুন
    duplicates = []
    paths      = list(loaded.keys())
    
    for i in tqdm(range(len(paths)), desc='Comparing pairs'):
        for j in range(i + 1, len(paths)):
            score, _ = ssim(loaded[paths[i]], loaded[paths[j]], full=True)
            if score > threshold:
                duplicates.append((paths[i], paths[j], score))
    
    return duplicates


def remove_duplicates(class_info, threshold=0.95):
    """
    Duplicate image এবং তার mask remove করে
    Clean image list return করে
    """
    clean_data = {}  # class → list of (image_path, mask_path)
    total_removed = 0
    
    for cls, info in class_info.items():
        print(f'\n📂 Processing class: {cls.upper()}')
        images = info['images']
        
        # Duplicate খুঁজুন
        duplicates = find_duplicates(images, threshold=threshold)
        
        # কোন image remove করব সেটা ঠিক করুন
        to_remove = set()
        for f1, f2, score in duplicates:
            if f1 not in to_remove:  # first টা রাখব, second টা remove করব
                to_remove.add(f2)
        
        # Clean list তৈরি করুন
        clean_pairs = []
        for img_path in images:
            if str(img_path) not in to_remove:
                mask_path = img_path.parent / (img_path.stem + '_mask' + img_path.suffix)
                clean_pairs.append({
                    'image': str(img_path),
                    'mask':  str(mask_path) if mask_path.exists() else None,
                    'label': cls
                })
        
        clean_data[cls] = clean_pairs
        total_removed += len(to_remove)
        
        print(f'  Original : {len(images)}')
        print(f'  Removed  : {len(to_remove)}')
        print(f'  Remaining: {len(clean_pairs)}')
        
        if duplicates:
            print(f'  Found duplicates (SSIM > {threshold}):')
            for f1, f2, score in duplicates[:3]:  # max 3 দেখাও
                print(f'    {Path(f1).name} ↔ {Path(f2).name} (score={score:.3f})')
    
    print(f'\n✅ Total duplicates removed: {total_removed}')
    return clean_data


# Run করুন
clean_data = remove_duplicates(class_info, threshold=0.95)

In [ ]:
# Clean dataset এর count দেখুন
print('📊 Clean Dataset Summary:')
print('=' * 35)
total = 0
for cls, pairs in clean_data.items():
    print(f'  {cls.upper():12s}: {len(pairs)} images')
    total += len(pairs)
print(f'  {"TOTAL":12s}: {total} images')

## Step 5: Train / Val / Test Split (70 / 15 / 15)

> **Stratified split** — প্রতিটি class থেকে সমানভাবে নেওয়া হবে

In [ ]:
def stratified_split(clean_data, train_ratio=0.70, val_ratio=0.15, seed=42):
    """
    প্রতিটি class থেকে stratified split করে
    Returns: train_list, val_list, test_list
    """
    random.seed(seed)
    
    train_list, val_list, test_list = [], [], []
    
    print('📂 Splitting dataset...')
    print('=' * 40)
    
    for cls, pairs in clean_data.items():
        shuffled = pairs.copy()
        random.shuffle(shuffled)
        
        n         = len(shuffled)
        n_train   = int(n * train_ratio)
        n_val     = int(n * val_ratio)
        # বাকিটা test
        
        train_list.extend(shuffled[:n_train])
        val_list.extend(shuffled[n_train:n_train + n_val])
        test_list.extend(shuffled[n_train + n_val:])
        
        print(f'  {cls.upper():12s} → Train: {n_train:3d} | Val: {n_val:3d} | Test: {n - n_train - n_val:3d}')
    
    print(f'  {"TOTAL":12s} → Train: {len(train_list):3d} | Val: {len(val_list):3d} | Test: {len(test_list):3d}')
    return train_list, val_list, test_list


train_list, val_list, test_list = stratified_split(clean_data)

In [ ]:
# Split visualization
label_map = {'normal': 0, 'benign': 1, 'malignant': 2}
splits    = {'Train': train_list, 'Val': val_list, 'Test': test_list}
classes   = ['normal', 'benign', 'malignant']
colors    = ['#2ecc71', '#e74c3c', '#3498db']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (split_name, split_data) in zip(axes, splits.items()):
    counts = {cls: sum(1 for d in split_data if d['label'] == cls) for cls in classes}
    ax.bar(classes, [counts[c] for c in classes], color=colors, edgecolor='black')
    ax.set_title(f'{split_name} Set (n={len(split_data)})', fontsize=12, fontweight='bold')
    ax.set_ylabel('Count')
    for i, cls in enumerate(classes):
        ax.text(i, counts[cls] + 0.3, str(counts[cls]), ha='center', fontsize=10)

plt.suptitle('Train / Val / Test Class Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('split_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Split visualization saved!')

In [ ]:
# Split JSON হিসেবে save করুন — পরে Week 2 তে use করব
split_info = {
    'train': train_list,
    'val':   val_list,
    'test':  test_list
}

with open('busi_split.json', 'w') as f:
    json.dump(split_info, f, indent=2)

print('✅ Split saved as busi_split.json')
print('   Week 2 এ এই file টা load করব।')

## Step 6: Image Statistics দেখুন (Size, Pixel Distribution)

In [ ]:
def analyze_image_stats(data_list, sample_size=50):
    """
    Image size এবং pixel intensity distribution দেখায়
    """
    sample  = random.sample(data_list, min(sample_size, len(data_list)))
    heights, widths, mean_pixels = [], [], []
    
    for item in tqdm(sample, desc='Analyzing images'):
        img = cv2.imread(item['image'], cv2.IMREAD_GRAYSCALE)
        if img is not None:
            heights.append(img.shape[0])
            widths.append(img.shape[1])
            mean_pixels.append(img.mean())
    
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    axes[0].hist(heights, bins=20, color='#3498db', edgecolor='black')
    axes[0].set_title('Image Height Distribution', fontweight='bold')
    axes[0].set_xlabel('Height (pixels)')
    axes[0].axvline(np.mean(heights), color='red', linestyle='--', label=f'Mean={np.mean(heights):.0f}')
    axes[0].legend()
    
    axes[1].hist(widths, bins=20, color='#2ecc71', edgecolor='black')
    axes[1].set_title('Image Width Distribution', fontweight='bold')
    axes[1].set_xlabel('Width (pixels)')
    axes[1].axvline(np.mean(widths), color='red', linestyle='--', label=f'Mean={np.mean(widths):.0f}')
    axes[1].legend()
    
    axes[2].hist(mean_pixels, bins=20, color='#e74c3c', edgecolor='black')
    axes[2].set_title('Mean Pixel Intensity', fontweight='bold')
    axes[2].set_xlabel('Mean pixel value (0-255)')
    
    plt.suptitle('Image Statistics (Training Set Sample)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('image_stats.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'📏 Height  → Mean: {np.mean(heights):.0f} | Min: {min(heights)} | Max: {max(heights)}')
    print(f'📏 Width   → Mean: {np.mean(widths):.0f}  | Min: {min(widths)} | Max: {max(widths)}')
    print(f'\n💡 Recommendation: Resize সব image কে 256×256 বা 512×512 তে')

analyze_image_stats(train_list)

## Step 7: Augmentation Pipeline তৈরি করুন

> **Albumentations** library use করব — medical image augmentation এর জন্য সবচেয়ে popular

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGE_SIZE = 256  # সব image কে এই size এ resize করব

# ── Training Augmentation ──────────────────────────────────────────
train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    
    # Geometric augmentations
    A.HorizontalFlip(p=0.5),                          # আনুভূমিক flip
    A.VerticalFlip(p=0.3),                            # উল্লম্ব flip
    A.RandomRotate90(p=0.3),                          # 90° rotation
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.1,
        rotate_limit=15,
        p=0.4
    ),
    
    # Intensity augmentations (ultrasound এর জন্য উপযুক্ত)
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),      # Noise যোগ করুন
    A.GaussianBlur(blur_limit=3, p=0.2),               # Slight blur
    A.CLAHE(clip_limit=2.0, p=0.3),                   # Contrast enhancement
    
    # Normalize এবং Tensor convert
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    ToTensorV2()
], additional_targets={'mask': 'mask'})  # Image এবং mask একসাথে transform


# ── Validation/Test Augmentation (শুধু resize + normalize) ───────
val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    ToTensorV2()
], additional_targets={'mask': 'mask'})


print('✅ Augmentation pipelines defined!')
print(f'   Image size: {IMAGE_SIZE}×{IMAGE_SIZE}')
print(f'   Train transforms: {len(train_transform.transforms)}')
print(f'   Val transforms  : {len(val_transform.transforms)}')

In [ ]:
# Augmentation দেখুন — একটা sample image এ apply করে
def visualize_augmentations(data_list, n_augmentations=5):
    """
    একটা image এ বিভিন্ন augmentation দেখায়
    """
    # একটা benign image নিন
    sample_item = next((d for d in data_list if d['label'] == 'benign'), data_list[0])
    
    img  = cv2.imread(sample_item['image'])
    img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    if sample_item['mask'] and os.path.exists(sample_item['mask']):
        mask = cv2.imread(sample_item['mask'], cv2.IMREAD_GRAYSCALE)
    
    # Augmentation pipeline (tensor ছাড়া — visualization এর জন্য)
    aug_pipeline = A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.HorizontalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=20, p=0.7),
        A.RandomBrightnessContrast(p=0.5),
        A.GaussNoise(p=0.3),
    ], additional_targets={'mask': 'mask'})
    
    fig, axes = plt.subplots(2, n_augmentations + 1, figsize=((n_augmentations + 1) * 3, 6))
    
    # Original দেখান
    orig_resized = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
    mask_resized = cv2.resize(mask, (IMAGE_SIZE, IMAGE_SIZE))
    axes[0][0].imshow(orig_resized)
    axes[0][0].set_title('Original', fontweight='bold')
    axes[0][0].axis('off')
    axes[1][0].imshow(mask_resized, cmap='gray')
    axes[1][0].set_title('Original Mask', fontweight='bold')
    axes[1][0].axis('off')
    
    # Augmented versions দেখান
    for i in range(1, n_augmentations + 1):
        augmented = aug_pipeline(image=img, mask=mask)
        axes[0][i].imshow(augmented['image'])
        axes[0][i].set_title(f'Aug {i}', fontweight='bold')
        axes[0][i].axis('off')
        axes[1][i].imshow(augmented['mask'], cmap='gray')
        axes[1][i].set_title(f'Mask {i}', fontweight='bold')
        axes[1][i].axis('off')
    
    plt.suptitle('Augmentation Examples (Image + Mask)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('augmentation_examples.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_augmentations(train_list)

## Step 8: PyTorch Dataset Class তৈরি করুন

> এটা Week 2 তে সরাসরি use করব — model training এর সময়

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

LABEL_MAP = {'normal': 0, 'benign': 1, 'malignant': 2}

class BUSIDataset(Dataset):
    """
    BUSI Dataset — Image + Mask + Label একসাথে return করে
    Week 2 এর model training এ সরাসরি use করব
    """
    
    def __init__(self, data_list, transform=None):
        """
        Args:
            data_list : list of dicts {'image', 'mask', 'label'}
            transform : albumentations transform pipeline
        """
        self.data      = data_list
        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Image load করুন (RGB)
        image = cv2.imread(item['image'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Mask load করুন (grayscale, binary)
        if item['mask'] and os.path.exists(item['mask']):
            mask = cv2.imread(item['mask'], cv2.IMREAD_GRAYSCALE)
            mask = (mask > 127).astype(np.uint8)  # Binary করুন
        else:
            mask = np.zeros(image.shape[:2], dtype=np.uint8)  # Empty mask
        
        # Label
        label = LABEL_MAP[item['label']]
        
        # Transform apply করুন
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image     = augmented['image']  # Tensor [3, H, W]
            mask      = augmented['mask']   # Tensor [H, W]
        
        # Mask dimension যোগ করুন [1, H, W]
        if isinstance(mask, torch.Tensor):
            mask = mask.unsqueeze(0).float()
        
        return {
            'image': image,                          # [3, 256, 256]
            'mask':  mask,                           # [1, 256, 256]
            'label': torch.tensor(label, dtype=torch.long)
        }


# Dataset তৈরি করুন
train_dataset = BUSIDataset(train_list, transform=train_transform)
val_dataset   = BUSIDataset(val_list,   transform=val_transform)
test_dataset  = BUSIDataset(test_list,  transform=val_transform)

# DataLoader তৈরি করুন
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False, num_workers=2)

print('✅ Dataset and DataLoaders ready!')
print(f'   Train batches : {len(train_loader)}')
print(f'   Val batches   : {len(val_loader)}')
print(f'   Test batches  : {len(test_loader)}')

In [ ]:
# একটা batch নিয়ে shape verify করুন
batch = next(iter(train_loader))

print('📦 Sample Batch Info:')
print(f'   Image shape : {batch["image"].shape}')   # [B, 3, 256, 256]
print(f'   Mask shape  : {batch["mask"].shape}')    # [B, 1, 256, 256]
print(f'   Label shape : {batch["label"].shape}')   # [B]
print(f'   Labels      : {batch["label"].tolist()}')
print()
print('✅ সব ঠিক আছে! Week 2 তে model training শুরু করা যাবে।')

## Step 9: সব কিছু Google Drive এ Save করুন

In [ ]:
# সব output Google Drive এ save করুন
SAVE_DIR = '/content/drive/MyDrive/BUSI_Project/week1_outputs'
os.makedirs(SAVE_DIR, exist_ok=True)

# JSON split file copy করুন
shutil.copy('busi_split.json', f'{SAVE_DIR}/busi_split.json')

# Charts copy করুন
for chart in ['class_distribution.png', 'split_distribution.png',
              'sample_images.png', 'image_stats.png', 'augmentation_examples.png']:
    if os.path.exists(chart):
        shutil.copy(chart, f'{SAVE_DIR}/{chart}')

print(f'✅ সব files save হয়েছে: {SAVE_DIR}')
print()
print('📋 Week 1 Complete! Summary:')
print('=' * 40)
print(f'  ✅ Dataset explore করা হয়েছে')
print(f'  ✅ Duplicates remove করা হয়েছে')
print(f'  ✅ Train/Val/Test split done (70/15/15)')
print(f'  ✅ Augmentation pipeline ready')
print(f'  ✅ PyTorch Dataset class ready')
print(f'  ✅ DataLoaders ready')
print()
print('🚀 Week 2 তে: U-Net segmentation এবং ResNet18 classification baseline train করব!')

---
## ✅ Week 1 Complete!

### Week 2 তে যা করব:
```
Week 2: Baseline train করুন
├── U-Net segmentation only (Dice score দেখুন)
└── ResNet18 classification only (Accuracy, F1 দেখুন)
```

**Save করা files:**
- `busi_split.json` → Train/Val/Test split info (Week 2 তে load করব)
- `class_distribution.png` → Class balance visualization
- `split_distribution.png` → Split visualization  
- `augmentation_examples.png` → Augmentation examples
- `image_stats.png` → Image size statistics